In [1]:
# ===== درس ۱۰: مسیر فایل و ذخیره/بارگذاری =====

# ۱. اتصال Google Drive
from google.colab import drive
drive.mount('/content/drive')

# ۲. تعریف مسیرهای پروژه با pathlib
from pathlib import Path

PROJECT = Path("/content/drive/MyDrive/scRNA_project")
LESSON = PROJECT / "lesson10"

CODE = LESSON / "code"
NOTES = LESSON / "notes"
OUTPUT = LESSON / "output"

# ۳. ساخت پوشه‌های همین درس
for folder in [CODE, NOTES, OUTPUT]:
    folder.mkdir(parents=True, exist_ok=True)
    print("آماده شد:", folder)

# ۴. نصب و بارگذاری کتابخانه‌ها + دیتاست PBMC-3k از صفر
!pip install scanpy -q
import scanpy as sc
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

adata = sc.datasets.pbmc3k()

Mounted at /content/drive
آماده شد: /content/drive/MyDrive/scRNA_project/lesson10/code
آماده شد: /content/drive/MyDrive/scRNA_project/lesson10/notes
آماده شد: /content/drive/MyDrive/scRNA_project/lesson10/output
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 33.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.8/188.8 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.0/40.0 kB 2.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.9/10.9 MB 92.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.7/363.7 kB 24.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 93.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 4.3 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the p

  0%|          | 0.00/5.58M [00:00<?, ?B/s]

In [2]:
# ۵. تکرار QC درس ۹ + فیلتر تکمیلی (آستانه‌ی بالای ژن، برای doublet)
adata.var['mt'] = adata.var_names.str.startswith('MT-')
sc.pp.calculate_qc_metrics(adata, qc_vars=['mt'], percent_top=None, log1p=False, inplace=True)

sc.pp.filter_cells(adata, min_genes=200)
sc.pp.filter_genes(adata, min_cells=3)

# فیلتر نهایی: هم mt پایین، هم ژن غیرعادی‌بالا (doublet) نداشته باشه
adata = adata[
    (adata.obs['pct_counts_mt'] < 5) &
    (adata.obs['n_genes_by_counts'] < 2500),
    :
].copy()

print("shape after full QC:", adata.shape)

shape after full QC: (2638, 13714)


In [3]:
# ۶. پوشه‌ی مرکزی برای نگهداری نسخه‌های تمیزشده‌ی داده
DATASETS = PROJECT / "datasets"
DATASETS.mkdir(parents=True, exist_ok=True)

save_path = DATASETS / "pbmc3k_filtered.h5ad"
print(save_path)

/content/drive/MyDrive/scRNA_project/datasets/pbmc3k_filtered.h5ad


In [4]:
# ۷. ذخیره‌ی adata روی فایل با فرمت h5ad
adata.write(str(save_path))
print("ذخیره شد.")

ذخیره شد.


In [5]:
# ۸. بارگذاری دوباره از روی فایل ذخیره‌شده، برای اطمینان از سلامت فایل
adata_reloaded = sc.read_h5ad(str(save_path))

print("shape:", adata_reloaded.shape)
print(adata_reloaded.obs.columns.tolist())
print(adata_reloaded.obs[['n_genes_by_counts', 'pct_counts_mt']].head())

shape: (2638, 13714)
['n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes']
                  n_genes_by_counts  pct_counts_mt
index                                             
AAACATACAACCAC-1                781       3.015283
AAACATTGAGCTAC-1               1352       3.793596
AAACATTGATCAGC-1               1131       0.889171
AAACCGTGCTTCCG-1                960       1.743085
AAACCGTGTATGCG-1                522       1.223242
